# Off-road segmentation — training (v2) and evaluation on Colab

Runtime → Change runtime type → **T4 GPU**. Run the cells in order.

Two trainings with the same v2 recipe: **ResNet34 + U-Net** (the exam architecture) and **SegFormer-B2** (Cityscapes pre-trained). Each takes about 1.5–2 hours; they can be run in different sessions. Checkpoints are saved on Drive at every epoch: after a disconnection, run the setup cells again and re-run the training cell, it resumes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Paths — edit these

In [ ]:
CODE_ZIP         = '/content/drive/MyDrive/offroad-terrain-segmentation.zip'
TRAIN_DATA       = '/content/drive/MyDrive/ProjectWorkML_Group70/clear_train'    # 896 cleaned samples: folder or .zip
BASELINE_WEIGHTS = '/content/drive/MyDrive/ProjectWorkML_Group70/final_resnet34_unet_model.pth'
YCOR_ARCHIVE     = '/content/drive/MyDrive/ycor/yamaha_v0.tar.gz'   # official YCOR archive as downloaded (optional)
OUT              = '/content/drive/MyDrive/offroad_seg_runs'                      # results and checkpoints

import os, json, glob
for p in [CODE_ZIP, TRAIN_DATA, BASELINE_WEIGHTS, YCOR_ARCHIVE]:
    print('OK     ' if os.path.exists(p) else 'MISSING', p)
os.makedirs(OUT, exist_ok=True)

In [ ]:
!rm -rf /content/repo && mkdir -p /content/repo && unzip -q "{CODE_ZIP}" -d /content/repo
# local copy of the data: much faster than reading from Drive during training
!mkdir -p /content/data
if TRAIN_DATA.endswith('.zip'):
    if not os.path.isdir('/content/data/train'):
        !mkdir -p /content/tmp_train && unzip -q "{TRAIN_DATA}" -d /content/tmp_train
        # the folder that contains the sample sub-folders
        root = os.path.dirname(os.path.dirname(sorted(glob.glob('/content/tmp_train/**/labels.png', recursive=True))[0]))
        !mv "{root}" /content/data/train
else:
    !rsync -a "{TRAIN_DATA}/" /content/data/train/
# YCOR: extract the archive and keep its 'valid' folder (sub-folders with rgb.jpg and labels.png)
if os.path.exists(YCOR_ARCHIVE) and not os.path.isdir('/content/data/ycor_valid'):
    !mkdir -p /content/ycor && tar -xzf "{YCOR_ARCHIVE}" -C /content/ycor
    valid = [d for d in glob.glob('/content/ycor/**/valid', recursive=True) if glob.glob(f'{d}/*/labels.png')]
    print('YCOR valid folder:', valid)
    if valid:
        !mv "{valid[0]}" /content/data/ycor_valid && ls /content/data/ycor_valid | wc -l
!pip install -q transformers
%cd /content/repo
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!ls /content/data/train | wc -l

## 2. Baseline (exam model) on its own validation split
Re-creates the original stratified split (same seed) and evaluates the original weights with the official per-image metric.

In [ ]:
from offroad_seg.data import list_samples, stratified_split
tr, va = stratified_split('/content/data/train', list_samples('/content/data/train'), val_size=180, seed=23)
json.dump({'train': tr, 'val': va}, open(f'{OUT}/baseline_split.json', 'w'))
!python evaluate.py --weights "{BASELINE_WEIGHTS}" --data-root /content/data/train --split "{OUT}/baseline_split.json" --output "{OUT}/eval_baseline_val.json"

## 3a. Train v2 — ResNet34 + U-Net (≈1.5–2 h)

In [ ]:
!python train.py --recipe v2 --model resnet34_unet --data-root /content/data/train --output-dir "{OUT}/v2_resnet34" --batch-size 4 --num-workers 2

## 3b. Train v2 — SegFormer-B2 with mixed precision (≈1–1.5 h, can be run in another session)

In [ ]:
# --amp (mixed precision) keeps training memory below the 5 GB limit of the course
!python train.py --recipe v2 --model segformer-b2 --amp --data-root /content/data/train --output-dir "{OUT}/v2_segformer_b2" --batch-size 4 --num-workers 2

## 4. Evaluation on the v2 validation split (no near-duplicates of training images)
Both v2 runs use the same split. The baseline is evaluated on it too, for reference only: it saw some of these images during training.

In [ ]:
for run in ['v2_resnet34', 'v2_segformer_b2']:
    if os.path.exists(f'{OUT}/{run}/best_model.pth'):
        !python evaluate.py --weights "{OUT}/{run}/best_model.pth" --data-root /content/data/train --split "{OUT}/{run}/split.json" --output "{OUT}/eval_{run}_val.json"
split = f'{OUT}/v2_resnet34/split.json' if os.path.exists(f'{OUT}/v2_resnet34/split.json') else f'{OUT}/v2_segformer_b2/split.json'
!python evaluate.py --weights "{BASELINE_WEIGHTS}" --data-root /content/data/train --split "{split}" --output "{OUT}/eval_baseline_on_v2val.json"

## 5. All models on the official YCOR validation set
145 images that no model has seen: the fairest comparison.

In [ ]:
if os.path.isdir('/content/data/ycor_valid'):
    !python evaluate.py --weights "{BASELINE_WEIGHTS}" --data-root /content/data/ycor_valid --output "{OUT}/eval_baseline_ycor.json"
    for run in ['v2_resnet34', 'v2_segformer_b2']:
        if os.path.exists(f'{OUT}/{run}/best_model.pth'):
            !python evaluate.py --weights "{OUT}/{run}/best_model.pth" --data-root /content/data/ycor_valid --output "{OUT}/eval_{run}_ycor.json" --examples-dir "{OUT}/examples_{run}_ycor"
else:
    print('YCOR validation set not found: skipped')

## 6. Summary

In [ ]:
for f in sorted(glob.glob(f'{OUT}/eval_*.json')):
    r = json.load(open(f))
    print(f"{os.path.basename(f):36} per-image mIoU {r['mean_iou_per_image']:.4f} | dataset mIoU {r['mean_iou_dataset']:.4f} | {r['fps']:.1f} FPS | {r['peak_gpu_memory_gb'] or 0:.2f} GB")
!cd "{OUT}" && rm -f results.zip && zip -q -r results.zip eval_*.json */split.json */class_weights.npy examples_* 2>/dev/null; ls -la "{OUT}"